# Estonian Radio Name Mentions
Count how often the top-20 candidates from the 2025 KOV elections are mentioned across Vikerraadio audio files

## Install dependencies

In [1]:
!pip install -q faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 50.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 47.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 74.3 MB/s eta 0:00:00:00:0100:01


## Worker.py

Transcription worker saved to disk via `%%writefile`. Loads faster-whisper, extracts 15-second snippets every 30 minutes, transcribes in Estonian, and saves one .json per file. Skips already-processed files

In [2]:
%%writefile worker.py
import json, subprocess
import numpy as np
from pathlib import Path

SAMPLE_EVERY_MS = 20 * 60 * 1000
SNIPPET_MS      = 20 * 1000

def get_duration_ms(path):
    cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration",
           "-of", "json", str(path)]
    out = subprocess.check_output(cmd)
    return int(float(json.loads(out)["format"]["duration"]) * 1000)

def load_audio_segment(path, start_ms, duration_ms):
    cmd = ["ffmpeg", "-ss", str(start_ms/1000), "-t", str(duration_ms/1000),
           "-i", str(path), "-f", "s16le", "-ac", "1",
           "-acodec", "pcm_s16le", "-ar", "16000", "-"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    audio_bytes = proc.stdout.read()
    return np.frombuffer(audio_bytes, np.int16).astype(np.float32) / 32768.0

def process_files(gpu_id, file_list, out_dir_str, total_files):
    import torch
    from faster_whisper import WhisperModel
    out_dir = Path(out_dir_str)
    device  = "cuda" if torch.cuda.is_available() else "cpu"
    compute = "float16" if device == "cuda" else "int8"
    print(f"[GPU {gpu_id}] Loading faster-whisper base on {device}...")
    model = WhisperModel("base", device=device, compute_type=compute)

    for fpath in file_list:
        fpath    = Path(fpath)
        out_file = out_dir / (fpath.stem + ".json")
        if out_file.exists():
            continue
        snippets = []
        try:
            duration_ms = get_duration_ms(fpath)
            starts      = list(range(0, duration_ms, SAMPLE_EVERY_MS))
            for j, start_ms in enumerate(starts):
                audio = load_audio_segment(fpath, start_ms, SNIPPET_MS)
                print(f"[GPU {gpu_id}] snippet {j+1}/{len(starts)} ({round(start_ms/60000,1)} min)", end="\r")
                segments, _ = model.transcribe(
                    audio,
                    language="et",
                    beam_size=1,
                    vad_filter=True,
                )
                text = " ".join(seg.text.strip() for seg in segments)
                snippets.append({"start_min": round(start_ms/60000, 1), "text": text})
            out_file.write_text(json.dumps(
                {"file": fpath.name, "snippets": snippets},
                ensure_ascii=False, indent=2
            ))
            done      = len(list(out_dir.glob("*.json")))
            remaining = total_files - done
            print(f"[GPU {gpu_id}] {fpath.name} | ready: {done}/{total_files} | remain: {remaining}", flush=True)
        except Exception as e:
            print(f"[GPU {gpu_id}] ERROR {fpath.name}: {e}", flush=True)

Writing worker.py


## Run transcription

Imports process_files from worker.py, scans for `.mp3`/`.aac` files, skips already-done ones, and runs transcription. Splits across 2 GPUs if available. Autosaves a zip backup

In [3]:
import sys, json, os
import torch
import multiprocessing as mp
import threading
import shutil
import time
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
from worker import process_files

print("Input dirs:", os.listdir("/kaggle/input"))

audio_dir = Path("/kaggle/input/datasets/nastya2604/vikerraadio/vikerraadio")
all_files = list(audio_dir.rglob("*.mp3")) + list(audio_dir.rglob("*.aac"))
print(f"Files found: {len(all_files)}")

out_dir = Path("/kaggle/working/transcripts")
out_dir.mkdir(exist_ok=True)

restored = len(list(out_dir.glob("*.json")))
print(f"Already done: {restored}")

pending = [str(f) for f in all_files if not (out_dir / (f.stem + ".json")).exists()]
print(f"Pending: {len(pending)}")

def autosave(out_dir, interval=300):
    while True:
        time.sleep(interval)
        try:
            shutil.make_archive("/kaggle/working/transcripts_backup", "zip", str(out_dir))
            done = len(list(Path(out_dir).glob("*.json")))
            print(f"Autosave: {done} files to transcripts_backup.zip", flush=True)
        except Exception as e:
            print(f"Autosave error: {e}", flush=True)

threading.Thread(target=autosave, args=(out_dir,), daemon=True).start()
print("Autosave started")

n_gpus = torch.cuda.device_count()
print(f"GPUs: {n_gpus}")

if n_gpus >= 2 and len(pending) > 0:
    half = len(pending) // 2
    splits = [pending[:half], pending[half:]]
    print(f"GPU0: {len(splits[0])} | GPU1: {len(splits[1])}")
    ctx = mp.get_context("spawn")
    with ctx.Pool(2) as pool:
        pool.starmap(process_files, [
            (0, splits[0], str(out_dir), len(all_files)),
            (1, splits[1], str(out_dir), len(all_files)),
        ])
elif len(pending) > 0:
    process_files(0, pending, str(out_dir), len(all_files))

print("Done")

Input dirs: ['datasets']
Files found: 2883
Already done: 0
Pending: 2883
Autosave started
GPUs: 2
GPU0: 1441 | GPU1: 1442


[GPU 0] Loading faster-whisper base on cuda...
[GPU 0] 2025-10-24_11.mp3 | ready: 1/2883 | remain: 2882
[GPU 1] Loading faster-whisper base on cuda...
[GPU 1] 2025-07-02_21.mp3 | ready: 2/2883 | remain: 2881
[GPU 0] 2025-07-23_20.mp3 | ready: 3/2883 | remain: 2880
[GPU 1] 2025-08-09_08.mp3 | ready: 4/2883 | remain: 2879
[GPU 0] 2025-10-22_09.mp3 | ready: 5/2883 | remain: 2878
[GPU 0] 2025-07-24_04.mp3 | ready: 6/2883 | remain: 2877
[GPU 1] 2025-09-08_12.mp3 | ready: 7/2883 | remain: 2876
[GPU 1] 2025-07-28_11.mp3 | ready: 8/2883 | remain: 2875
[GPU 0] 2025-10-04_15.mp3 | ready: 9/2883 | remain: 2874
[GPU 1] 2025-07-19_10.mp3 | ready: 10/2883 | remain: 2873
[GPU 0] 2025-09-12_09.mp3 | ready: 11/2883 | remain: 2872
[GPU 1] 2025-10-10_18.mp3 | ready: 12/2883 | remain: 2871
[GPU 0] 2025-10-23_05.mp3 | ready: 13/2883 | remain: 2870
[GPU 1] 2025-08-30_05.mp3 | ready: 14/2883 | remain: 2869
[GPU 0] 2025-08-24_15.mp3 | ready: 15/2883 | remain: 2868
[GPU 0] 2025-07-26_04.mp3 | ready: 16/2883 | 

## Fetch top-20 candidates

Fetches live 2025 KOV election results, picks the 20 candidates with the most votes, and saves them to `top20.csv`

In [4]:
import requests, xml.etree.ElementTree as ET, pandas as pd

url  = "https://opendata.valimised.ee/api/KOV_2025/RESULTS.xml"
root = ET.fromstring(requests.get(url).content)
ns   = "https://opendata.valimised.ee/schemas/election-result/kov/v2/"

candidates = []
for cand in root.iter(f"{{{ns}}}candidate"):
    forename = cand.findtext(f"{{{ns}}}forename")
    surname  = cand.findtext(f"{{{ns}}}surname")
    votes    = cand.findtext(f"{{{ns}}}votes")
    if forename and surname and votes:
        candidates.append({"name": f"{forename} {surname}", "votes": int(votes)})

df = (pd.DataFrame(candidates)
        .sort_values("votes", ascending=False)
        .head(20)
        .reset_index(drop=True))
df.index += 1
print(df.to_string())
df.to_csv("/kaggle/working/top20.csv", index=True, encoding="utf-8")
print("Saved to top20.csv")

                            name  votes
1                MIHHAIL KÕLVART  26526
2                    LAURI LAATS   9803
3   ANASTASSIA KOVALENKO-KÕLVART   7953
4                    URMAS KLAAS   5485
5                    TÕNIS LUKAS   4362
6                      JANA TOOM   4217
7             JEVGENI OSSINOVSKI   3390
8                  LAVLY PERLING   3155
9                  OLEG SILJANOV   2415
10                URMAS REINSALU   2302
11                  NATALIE METS   2265
12           ALEKSANDR TŠAPLÕGIN   2232
13                    MART HELME   2150
14            VADIM BELOBROVTSEV   2035
15                    KARMO KURI   2020
16                    JÜRI RATAS   1799
17                  MADLE LIPPUS   1777
18                    TANEL KIIK   1739
19                    ELO KIIVET   1731
20                    ILMAR RAAG   1700
Saved to top20.csv


## Count name mentions in transcripts

Loads top-20 candidates from `top20.csv`, scans all transcript `.json` files, and counts how often each name appears using fuzzy matching and phonetic similarity. Ambiguous surnames (e.g. "Mets", "Raag") require additional context to be counted. Saves results to `mentions.csv`.


In [5]:
!pip install -q python-Levenshtein fuzzy

import json, re, unicodedata, pandas as pd
from pathlib import Path
from collections import defaultdict
from Levenshtein import distance
import fuzzy

df_top20 = pd.read_csv("/kaggle/working/top20.csv")
TOP20 = df_top20["name"].tolist()

AMBIGUOUS = {"kuri", "mets", "toom", "raag", "laats"}

PERSON_HINTS = [
    "ütles", "sõnas", "lausus", "rääkis", "kommenteeris", "lisas", "märkis",
    "minister", "linnapea", "kandidaat", "poliitik", "saadik", "juht", "esimees",
    "karmo", "jana", "lauri", "natalie", "mart", "ilmar", "madle",
    "urmas", "tõnis", "jevgeni", "lavly", "oleg", "vadim", "tanel",
    "elo", "jüri", "mihhail", "anastassia", "aleksandr",
    "laats", "raag", "kuri",
]

dm = fuzzy.DMetaphone()

def ascii_fold(text):
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(ch)
    )

def build_variants(name):
    parts   = name.lower().split()
    surname = parts[-1] if parts else name.lower()
    variants = {
        name.lower(),
        ascii_fold(name.lower()),
        surname,
        ascii_fold(surname),
    }
    for part in parts:
        if len(part) >= 5:
            variants.add(part)
            variants.add(ascii_fold(part))
    return [v for v in variants if len(v) >= 5]

def phonetic_match(word, variant):
    try:
        w_codes = set(c for c in dm(word)    if c)
        v_codes = set(c for c in dm(variant) if c)
        return bool(w_codes & v_codes)
    except Exception:
        return False

def count_in_text(text, variants):
    txt         = text.lower()
    txt_ascii   = ascii_fold(txt)
    words       = txt.split()
    words_ascii = txt_ascii.split()
    count = 0
    for v in variants:
        has_diacritics = any(ord(ch) > 127 for ch in v)
        target_words   = words if has_diacritics else words_ascii
        threshold      = 1 if len(v) < 7 else 2
        v_ascii        = ascii_fold(v)
        for word in target_words:
            word_ascii = ascii_fold(word)
            if distance(word_ascii, v_ascii) <= threshold:
                count += 1
                break
    return count

def context_verify(text, surname, firstname):
    text_lower = text.lower()
    pos = text_lower.find(ascii_fold(surname.lower()))
    if pos == -1:
        pos = text_lower.find(surname.lower())
    if pos == -1:
        return False
    window = text_lower[max(0, pos-60) : pos+60]
    if firstname.lower() in window:
        return True
    return any(hint in window for hint in PERSON_HINTS)

name_variants = {name: build_variants(name) for name in TOP20}

total    = defaultdict(int)
verified = defaultdict(int)
transcript_dir = Path("/kaggle/working/transcripts")
files = list(transcript_dir.glob("*.json"))
print(f"Files to process: {len(files)}")

for i, jf in enumerate(files):
    try:
        data = json.loads(jf.read_text())
        for snippet in data["snippets"]:
            text = snippet.get("text", "")
            if not text:
                continue
            for name in TOP20:
                cnt = count_in_text(text, name_variants[name])
                if cnt:
                    total[name] += cnt
                    surname   = name.split()[-1].lower()
                    firstname = name.split()[0].lower()
                    if ascii_fold(surname) in AMBIGUOUS:
                        if context_verify(text, surname, firstname):
                            verified[name] += cnt
                    else:
                        verified[name] += cnt
    except Exception as e:
        print(f"Warning {jf.name}: {e}")
    if i % 100 == 0:
        print(f"  {i}/{len(files)} done...")

df_result = df_top20.copy()
df_result["mentions_raw"]      = df_result["name"].map(lambda n: total.get(n, 0))
df_result["mentions_verified"] = df_result["name"].map(lambda n: verified.get(n, 0))
df_result = df_result.sort_values("mentions_verified", ascending=False)

print("\n=== MENTIONS ===")
for _, row in df_result.iterrows():
    print(
        f"{row['name']:<35} "
        f"verified: {int(row['mentions_verified']):>4}  "
        f"raw: {int(row['mentions_raw']):>4}  "
        f"votes: {int(row['votes']):>6}"
    )

df_result = df_result[["name", "votes", "mentions_raw", "mentions_verified"]]
df_result.to_csv("/kaggle/working/mentions.csv", index=False, encoding="utf-8-sig")
print("Saved to mentions.csv")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 2.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 6.9 MB/s eta 0:00:0000:0100:01
Files to process: 2883
  0/2883 done...
  100/2883 done...
  200/2883 done...
  300/2883 done...
  400/2883 done...
  500/2883 done...
  600/2883 done...
  700/2883 done...
  800/2883 done...
  900/2883 done...
  1000/2883 done...
  1100/2883 done...
  1200/2883 done...
  1300/2883 done...
  1400/2883 done...
  1500/2883 done...
  1600/2883 done...
  1700/2883 done...
  1800/2883 done...
  1900/2883 done...
  2000/2883 done...
  2100/2883 done...
  2200/2883 done...
  2300/2883 done...
  2400/2883 done...
  2500/2883 done...
  2600/2883 done...
  2700/2883 done...
  2800/2883 done...

=== MENTIONS ===
TÕNIS LUKAS                         verified:   75  raw:   75  votes:   4362
URMAS KLAAS                         verified:   62  raw:   62  votes:   5485
MIHHAIL KÕLVART      

## Results - `mentions.csv`

`mentions_raw` - All fuzzy/phonetic matches found in transcripts

`mentions_verified` - Confirmed mentions only — name appears near a contextual marker (title, first name, speech verb)


`mentions_verified` is the primary metric. For candidates with common Estonian words as surnames (`Mets` = forest, `Raag` = branch, `Laats`, `Kuri`, `Toom`) raw matches are only counted if surrounding context confirms a person is being referenced — otherwise false positives are too high